# Citadel Agent Frameworks — three agents, three access contracts

Run **Access Contracts** first, then **Run All** here. The latest complete
`access-contracts.json` handoff for the selected azd gateway is picked up automatically;
there is no manifest-path copy/paste requirement. Exact deployed product, subscription,
grants, Key Vault secrets and Foundry connection metadata come from that handoff.

| Agent flavor | Agent created | Access contract | How it consumes the contract |
|---|---|---|---|
| Microsoft Agent Framework (`OpenAIChatClient.as_agent`) | Sales Assistant | Sales / Assistant / DEV (`LLM-Sales-Assistant-DEV` pattern) | Reads the contract's endpoint and key from Key Vault; calls Universal LLM with a framework session. |
| Microsoft Foundry SDK (`PromptAgentDefinition`) | Stable HR-ChatAgent; versions change only with its definition | HR / ChatAgent / DEV (`LLM-HR-ChatAgent-DEV` pattern) | Uses the contract's published Foundry project connection and ProjectManagedIdentity authentication. |
| LangChain (`ChatOpenAI`) | Support Bot | Support / Bot / DEV (`LLM-Support-Bot-DEV` pattern) | Uses that contract's exact APIM subscription key and Universal LLM endpoint, retaining conversation history. |

Actual IDs, which can include run-specific suffixes, are displayed after initialization.
Each agent performs a multi-turn conversation and validates remembered context.
Credentials are fetched afresh and never persisted in the manifest. Models must
be deployed, non-skipped, and allowed by the corresponding contract.

Microsoft Agent Framework and LangChain agents are constructed in the notebook
process; only the Foundry agent creates a remote versioned resource. Cleanup is
opt-in and never deletes the predecessor APIM contracts. Missing/disabled
predecessor integrations are reported explicitly rather than replaced with a
different subscription or connection.


## 1. Editable environment and predecessor inputs
Operations use the shared automatic assumptions unless explicitly overridden. Every agent intersects predecessor grants with eligible, non-skipped deployed models. Override names only to deployed, granted models.

### 1. Editable environment and predecessor inputs

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`manifest_path=None` automatically selects the latest complete Access Contracts handoff matching the selected gateway. Set `manifest_path` (or azd `VALIDATION_ACCESS_CONTRACTS_MANIFEST`) only to pin a particular run. Alternatively azd `VALIDATION_AGENT_CONTRACTS` supplies the same keyed exact records; no arbitrary subscription is selected.
It supplies the actual products, grants, Key Vault coordinates and Foundry
connection names; these cannot be inferred from unrelated backend registrations.
Example: `manifest_path = root / "validation" / ".runs" / "access-contracts" / "your-run-id" / "access-contracts.json"`.
`model_overrides` optionally pins a granted model, for example
`{"Sales-Assistant": "my-deployed-chat-model"}`; leave values `None` for selection.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Editable environment and predecessor inputs', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    from urllib.parse import quote
    from requests.structures import CaseInsensitiveDict
    from validationtools import ValidationContext, ValidationRun, ModelCatalog, RequestTransport
    from apimtools import APIMClientTool
    from contracttools import access_policy, prepare_artifact, deploy_contract
    from foundrychat import ensure_agent_version
    from consumerhelpers import (
        access_artifact, contract_identifiers, cli_json, chat_content, error_code,
        complete_backend_parameters, load_contract_manifest, find_access_contract_manifest, delete_owned_product,
        review_backend_artifact,
    )
    environment = None
    from_azd = True
    overrides = {}  # Explicit canonical config/capabilities override the selected azd environment.
    run_id = time.strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    cleanup_enabled = False

    # None automatically picks the latest complete handoff for the selected gateway.
    # Optional pin: root / "validation" / ".runs" / "access-contracts" / "your-run-id" / "access-contracts.json"
    manifest_path = None
    contract_records = {}  # Alternative: same keyed records as the handoff's "contracts".
    integration_overrides = {}  # Per stable name: KV secret metadata or exact Foundry endpoint/connection.
    model_overrides = {"Sales-Assistant": None, "HR-ChatAgent": None, "Support-Bot": None}
    turns = {
        "Sales-Assistant": ["Remember my company name: Contoso.", "Which company did I mention?"],
        "HR-ChatAgent": ["Remember my department: HR.", "Which department did I mention?"],
        "Support-Bot": ["Remember my VPN client runs on Windows.", "Which operating system did I mention?"],
    }
    memory_markers = {"Sales-Assistant": "contoso", "HR-ChatAgent": "hr", "Support-Bot": "windows"}
    run = ValidationRun("agent-frameworks")
    run_dir = root / "validation" / ".runs" / "agent-frameworks" / run_id
    sdk_metrics = {}
    owned_agent_version = None

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. One context and exact predecessor discovery
Automatically resolve the preceding Access Contracts handoff for this gateway, then validate its exact live products and model grants. Only the three intended subscription keys are read; no arbitrary subscription is selected.

### 2. One context and exact predecessor discovery

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. One context and exact predecessor discovery', inputs={'manifest_path': globals().get('manifest_path'), 'contract_records': globals().get('contract_records'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd'), 'integration_overrides': globals().get('integration_overrides')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    if not manifest_path and not contract_records:
        manifest_path = context.config.get("access_contracts_manifest") or context.config.get("VALIDATION_ACCESS_CONTRACTS_MANIFEST")
    if manifest_path:
        contract_records = load_contract_manifest(manifest_path)
    elif not contract_records:
        saved_records = context.config.get("agent_contracts") or context.config.get("VALIDATION_AGENT_CONTRACTS", {})
        try:
            contract_records = json.loads(saved_records) if isinstance(saved_records, str) else json.loads(json.dumps(saved_records))
        except json.JSONDecodeError as exc:
            raise ValueError("VALIDATION_AGENT_CONTRACTS must contain valid predecessor JSON.") from exc
    if not isinstance(contract_records, dict):
        raise ValueError("VALIDATION_AGENT_CONTRACTS must be a JSON object keyed by stable agent name.")
    if not manifest_path and not contract_records:
        manifest_path = find_access_contract_manifest(root, {
            "subscriptionId": context.config.get("subscription_id"),
            "resourceGroupName": context.config.get("resource_group"),
            "name": context.config.get("apim_name"),
        }, environment=context.config.get("environment"))
        if manifest_path is not None:
            contract_records = load_contract_manifest(manifest_path)
            utils.format_output("Access Contracts handoff", {"manifest": manifest_path}, part="INFO")
    for name, fields in integration_overrides.items():
        if name not in contract_records:
            raise ValueError("Integration override has no predecessor record.")
        contract_records[name].update(fields)
    mandatory_inputs["agent_frameworks.manifest_path or contract_records"] = True if manifest_path or contract_records else None
    for name in ("Sales-Assistant", "HR-ChatAgent", "Support-Bot"):
        record = contract_records.get(name, {})
        for field in ("allowed_models", "product_id", "subscription_name"):
            mandatory_inputs[f"contracts.{name}.{field}"] = record.get(field)
        capacity = record.get("capacity", {})
        for field in ("tokens_per_minute", "token_quota", "token_quota_period"):
            mandatory_inputs[f"contracts.{name}.capacity.{field}"] = capacity.get(field, record.get(field))
        if name == "Sales-Assistant":
            mandatory_inputs[f"contracts.{name}.use_keyvault"] = True if record.get("use_keyvault") else None
            for field in ("keyvault_name", "endpoint_secret", "apikey_secret"):
                mandatory_inputs[f"contracts.{name}.{field}"] = record.get(field)
        if name == "HR-ChatAgent":
            mandatory_inputs[f"contracts.{name}.use_foundry"] = True if record.get("use_foundry") else None
            for field in ("project_endpoint", "connection_name"):
                mandatory_inputs[f"contracts.{name}.{field}"] = record.get(field)
    try:
        utils.require_inputs(mandatory_inputs)
    except ValueError as exc:
        raise ValueError(
            f"{exc} Run Access Contracts first for this azd environment; its latest complete handoff "
            "is picked up automatically. Alternatively set manifest_path or "
            "VALIDATION_ACCESS_CONTRACTS_MANIFEST, or supply VALIDATION_AGENT_CONTRACTS / contract_records with exact product "
            "and subscription names, actual allowed_models/capacity and publication metadata. "
            "Backend registrations do not establish predecessor grants or Foundry connections."
        ) from exc
    utils.require_capabilities(context.catalog, list(dict.fromkeys(context.catalog.models_for("chat") + context.catalog.models_for("responses"))))
    for predecessor_name, predecessor_operation in [
        ("Sales-Assistant", "chat"), ("HR-ChatAgent", "responses"), ("Support-Bot", "chat")]:
        predecessor_catalog = ModelCatalog(context.config["llm_backends"], context.config["capabilities"],
            allowed_models=[item for item in contract_records[predecessor_name]["allowed_models"]
                            if item in context.catalog.models_for(predecessor_operation)], skipped_models=context.config.get("skipped_models", []), infer_operations=context.config.get("infer_operations", True))
        predecessor_catalog.select(predecessor_operation, override=model_overrides[predecessor_name])
    apim = APIMClientTool(context.config["resource_group"], context.config.get("apim_name") or "")
    apim.initialize_context(context.account, enumerate_subscriptions=False)
    transport = RequestTransport(timeout=60)
    utils.format_output('2. One context and exact predecessor discovery', {'value_1': "Environment:", 'value_2': environment or "(active)", 'value_3': "APIM:", 'value_4': apim.apim_resource_name}, status='info', part='RESULT')
    utils.format_output('2. One context and exact predecessor discovery', {'value_1': "Chat-capable configured models:", 'value_2': context.catalog.models_for("chat")}, status='info', part='RESULT')

    resolved = {}
    for name, operation in [("Sales-Assistant", "chat"), ("HR-ChatAgent", "responses"), ("Support-Bot", "chat")]:
        try:
            config = contract_records[name]
            if name == "Sales-Assistant" and not config.get("use_keyvault"):
                run.blocked(f"{name}/integration", "Enable Key Vault publication in the predecessor notebook.")
                raise ValueError("Required predecessor integration is disabled.")
            if name == "HR-ChatAgent" and not config.get("use_foundry"):
                run.blocked(f"{name}/integration", "Enable Foundry publication in the predecessor notebook.")
                raise ValueError("Required predecessor integration is incomplete.")
            needed = (["keyvault_name", "endpoint_secret", "apikey_secret"] if name == "Sales-Assistant"
                      else ["project_endpoint", "connection_name"] if name == "HR-ChatAgent" else [])
            missing = [field for field in needed if not config.get(field)]
            if missing:
                run.blocked(f"{name}/integration-metadata",
                            f"Supply predecessor publication metadata or integration_overrides: {missing}")
                raise ValueError("Required predecessor integration is incomplete.")
            coordinates = config.get("apim") or {}
            if coordinates and (
                coordinates.get("subscriptionId") != context.config["subscription_id"]
                or coordinates.get("resourceGroupName") != context.config["resource_group"]
                or coordinates.get("name") != apim.apim_resource_name
            ):
                raise ValueError("Predecessor manifest targets another APIM environment.")
            catalog = ModelCatalog(
                context.config["llm_backends"], context.config["capabilities"],
                allowed_models=[item for item in config["allowed_models"]
                                if item in context.catalog.models_for(operation)],
            skipped_models=context.config.get("skipped_models", []), infer_operations=context.config.get("infer_operations", True))
            model = catalog.select(operation, override=model_overrides[name])
            product_id = config["product_id"]
            subscription_name = config["subscription_name"]
            product = apim.client.product.get(context.config["resource_group"], apim.apim_resource_name, product_id)
            granted_apis = list(apim.client.product_api.list_by_product(
                context.config["resource_group"], apim.apim_resource_name, product_id,
            ))
            api_names = {api.name for api in granted_apis}
            run.check(f"{name}/api-grants", "universal-llm-api" in api_names,
                      expected="Universal LLM API granted", actual=sorted(api_names))
            if "universal-llm-api" not in api_names:
                raise ValueError("Required predecessor integration is incomplete.")
            policy = apim.client.product_policy.get(
                context.config["resource_group"], apim.apim_resource_name, product_id, "policy",
            )
            from xml.etree import ElementTree as ET
            policy_root = ET.fromstring(policy.value)
            grants = next((node.get("value", "") for node in policy_root.iter("set-variable")
                           if node.get("name") == "allowedModels"), "")
            model_granted = model in [item.strip() for item in grants.split(",")]
            limits = list(policy_root.iter("llm-token-limit"))
            run.check(f"{name}/model-grant", model_granted, expected=model, actual=grants)
            capacity = config.get("capacity", {})
            expected_limits = {
                "tokens-per-minute": capacity.get("tokens_per_minute", config.get("tokens_per_minute")),
                "token-quota": capacity.get("token_quota", config.get("token_quota")),
                "token-quota-period": capacity.get("token_quota_period", config.get("token_quota_period")),
            }
            if not all(value is not None for value in expected_limits.values()):
                run.blocked(f"{name}/capacity", "Predecessor manifest needs the configured capacity.")
                raise ValueError("Required predecessor integration is incomplete.")
            run.check(f"{name}/capacity",
                      any(all(node.get(key) == str(value) for key, value in expected_limits.items()) for node in limits),
                      expected=expected_limits, actual=[node.attrib for node in limits])
            if not model_granted:
                raise ValueError("Required predecessor integration is incomplete.")
            key = apim.get_subscription_key(subscription_name)
            resolved[name] = {"config": config, "model": model, "key": key}
            utils.format_output('2. One context and exact predecessor discovery', {'name': name, 'value_2': "selected:", 'model': model}, status='info', part='RESULT')
        except Exception as exc:
            run.blocked(f"{name}/preflight", f"Supply predecessor metadata and capabilities: {type(exc).__name__}: {exc}")
    models_api = apim.discover_api_exact("models")

    if set(resolved) != {"Sales-Assistant", "HR-ChatAgent", "Support-Bot"}:
        raise ValueError("All three predecessor contracts must pass initialization before SDK tests.")


    run_dir.mkdir(parents=True, exist_ok=True)

    utils.format_table("Agent to access-contract mapping", [{
        "agent": name, "framework": framework,
        "product": resolved[name]["config"]["product_id"],
        "subscription": resolved[name]["config"]["subscription_name"],
        "model": resolved[name]["model"], "connection method": connection,
    } for name, framework, connection in [
        ("Sales-Assistant", "Microsoft Agent Framework", "Key Vault endpoint/key"),
        ("HR-ChatAgent", "Foundry Prompt Agent", "Published project connection / ProjectManagedIdentity"),
        ("Support-Bot", "LangChain ChatOpenAI", "Exact APIM subscription key"),
    ]])


## 3. Microsoft Agent Framework + Key Vault
The predecessor must include **keyvault_name**, **endpoint_secret**, and **apikey_secret**.
`OpenAIChatClient.as_agent()` and `AgentSession` are the actual installed SDK APIs.
The saved endpoint must be the Universal LLM API root. Key Vault credentials must match
the intended predecessor subscription; SDK retries are disabled, not silently hidden.

### 3. Microsoft Agent Framework + Key Vault

Run the actual 3. Microsoft Agent Framework + Key Vault SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `resolved`, `item`, `config`, `turns`, `sdk_metrics`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('3. Microsoft Agent Framework + Key Vault', inputs={'resolved': globals().get('resolved'), 'item': globals().get('item'), 'config': globals().get('config'), 'turns': globals().get('turns'), 'sdk_metrics': globals().get('sdk_metrics')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    if "Sales-Assistant" in resolved:
        from azure.identity import DefaultAzureCredential
        from azure.keyvault.secrets import SecretClient
        from agent_framework import AgentSession
        from agent_framework.openai import OpenAIChatClient
        from openai import AsyncOpenAI
        item = resolved["Sales-Assistant"]
        try:
            config = item["config"]
            if not config.get("use_keyvault"):
                raise ValueError("Enable Key Vault publication on the Sales predecessor; no direct-key fallback.")
            vault = SecretClient(
                vault_url=f"https://{config['keyvault_name']}.vault.azure.net",
                credential=DefaultAzureCredential(),
            )
            endpoint = vault.get_secret(config["endpoint_secret"]).value
            key = vault.get_secret(config["apikey_secret"]).value
            run.check("sales/keyvault-publication", endpoint.rstrip("/") == models_api["url"].rstrip("/")
                      and key == item["key"], expected="Exact predecessor endpoint and credential", actual="resolved")
            if endpoint.rstrip("/") != models_api["url"].rstrip("/") or key != item["key"]:
                raise ValueError("Key Vault publication does not match the predecessor contract.")
            async with AsyncOpenAI(base_url=endpoint, api_key="unused",
                                   default_headers={"api-key": key}, max_retries=0) as client:
                chat_client = OpenAIChatClient(model=item["model"], async_client=client)
                sales_agent = chat_client.as_agent(name="SalesAssistant", instructions="You are a concise Sales Assistant.")
                session = AgentSession()
                for index, message in enumerate(turns["Sales-Assistant"], 1):
                    started = time.perf_counter()
                    result = await sales_agent.run(message, session=session)
                    text = result.text or ""
                    run.check(f"sales/turn-{index}", bool(text.strip()), expected="Nonempty framework output",
                              actual=text[:200], duration=time.perf_counter() - started, model=item["model"])
                    if index == 2:
                        run.check("sales/session-memory", memory_markers["Sales-Assistant"] in text.lower(),
                                  expected="Prior-turn company retained", actual=text[:200])
                    sdk_metrics[f"sales/turn-{index}"] = str(getattr(result, "usage", None) or "SDK did not expose usage")
        except Exception as exc:
            run.check("sales/sdk-invocation", False, expected="Key Vault + Agent Framework invocation",
                      actual=type(exc).__name__)
    else:
        run.blocked("sales/sdk-invocation", "Sales predecessor preflight did not complete.")


## 4. Foundry Prompt Agent + published project connection
The HR record needs **project_endpoint** and **connection_name**. Inspect the real connection,
then create or reuse the stable HR-ChatAgent using `connection_name/model`; only changed definitions publish a new version.
Responses use a persistent Foundry conversation. SDK usage, when exposed, is actual usage
and never an estimated token count.

### 4. Foundry Prompt Agent + published project connection

Run the actual 4. Foundry Prompt Agent + published project connection SDK example against the initialized contract and retain its returned content/session evidence.

**Input:** `resolved`, `item`, `config`, `models_api`, `turns`

**Output:** SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.


In [ ]:
with utils.notebook_step('4. Foundry Prompt Agent + published project connection', inputs={'resolved': globals().get('resolved'), 'item': globals().get('item'), 'config': globals().get('config'), 'models_api': globals().get('models_api'), 'turns': globals().get('turns')}, expected='SDK responses, session/agent state, and recorded feature validation; optional disabled paths remain explicit.', namespace=globals()):
    if "HR-ChatAgent" in resolved:
        from azure.identity import DefaultAzureCredential
        from azure.ai.projects import AIProjectClient
        from azure.ai.projects.models import PromptAgentDefinition
        item = resolved["HR-ChatAgent"]
        try:
            config = item["config"]
            if not config.get("use_foundry"):
                raise ValueError("Enable Foundry publication on the HR predecessor; no direct-key fallback.")
            with AIProjectClient(endpoint=config["project_endpoint"], credential=DefaultAzureCredential()) as project:
                connection = project.connections.get(config["connection_name"])
                target = getattr(connection, "target", None)
                run.check("hr/foundry-connection", bool(target) and target.rstrip("/") == models_api["url"].rstrip("/"),
                          expected=models_api["url"], actual=target or "(missing)")
                if not target or target.rstrip("/") != models_api["url"].rstrip("/"):
                    raise ValueError("Foundry connection targets a different gateway API.")
                agent, agent_version_created = ensure_agent_version(
                    project, agent_name="HR-ChatAgent",
                    definition=PromptAgentDefinition(
                        model=f"{config['connection_name']}/{item['model']}",
                        instructions="You are a concise HR assistant.",
                    ),
                )
                owned_agent_version = {
                    "project_endpoint": config["project_endpoint"], "name": agent.name, "version": agent.version,
                } if agent_version_created else None
                with project.get_openai_client(max_retries=0) as client:
                    conversation = client.conversations.create()
                    for index, message in enumerate(turns["HR-ChatAgent"], 1):
                        client.conversations.items.create(
                            conversation_id=conversation.id,
                            items=[{"type": "message", "role": "user", "content": message}],
                        )
                        started = time.perf_counter()
                        response = client.responses.create(
                            conversation=conversation.id,
                            extra_body={"agent_reference": {"name": agent.name, "version": agent.version, "type": "agent_reference"}},
                        )
                        text = response.output_text or ""
                        run.check(f"hr/turn-{index}", bool(text.strip()) and bool(response.id),
                                  expected="Nonempty Foundry agent response and ID", actual=text[:200],
                                  duration=time.perf_counter() - started, model=item["model"])
                        if index == 2:
                            run.check("hr/conversation-memory", memory_markers["HR-ChatAgent"] in text.lower(),
                                      expected="Prior-turn department retained", actual=text[:200])
                        sdk_metrics[f"hr/turn-{index}"] = str(response.usage or "SDK did not expose usage")
        except Exception as exc:
            run.check("hr/sdk-invocation", False, expected="Foundry connection and Prompt Agent invocation",
                      actual=type(exc).__name__)
    else:
        run.blocked("hr/sdk-invocation", "HR predecessor preflight did not complete.")


## 5. LangChain ChatOpenAI + direct predecessor credential
The actual LangChain model and message objects maintain conversation history; requests.post is not used as a stand-in.

### 5. LangChain ChatOpenAI + direct predecessor credential

Inspect or exercise 5. LangChain ChatOpenAI + direct predecessor credential using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `resolved`, `turns`, `response`, `index`, `sdk_metrics`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('5. LangChain ChatOpenAI + direct predecessor credential', inputs={'resolved': globals().get('resolved'), 'turns': globals().get('turns'), 'response': globals().get('response'), 'index': globals().get('index'), 'sdk_metrics': globals().get('sdk_metrics')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    if "Support-Bot" in resolved:
        from langchain_openai import ChatOpenAI
        from langchain_core.messages import SystemMessage, HumanMessage
        item = resolved["Support-Bot"]
        try:
            llm = ChatOpenAI(
                model=item["model"], base_url=models_api["url"], api_key="unused",
                default_headers={"api-key": item["key"]}, max_retries=0, timeout=60,
            )
            history = [SystemMessage(content="You are a concise technical support assistant.")]
            for index, message in enumerate(turns["Support-Bot"], 1):
                history.append(HumanMessage(content=message))
                started = time.perf_counter()
                response = llm.invoke(history)
                history.append(response)
                text = response.content
                run.check(f"support/turn-{index}", isinstance(text, str) and bool(text.strip()),
                          expected="Nonempty LangChain AIMessage", actual=str(text)[:200],
                          duration=time.perf_counter() - started, model=item["model"])
                if index == 2:
                    run.check("support/history-memory", isinstance(text, str) and memory_markers["Support-Bot"] in text.lower(),
                              expected="Prior-turn operating system retained", actual=str(text)[:200])
                sdk_metrics[f"support/turn-{index}"] = response.usage_metadata or "SDK did not expose usage"
        except Exception as exc:
            run.check("support/sdk-invocation", False, expected="LangChain invocation", actual=type(exc).__name__)
    else:
        run.blocked("support/sdk-invocation", "Support predecessor preflight did not complete.")


## 6. Evidence and strict verdict
Token estimates and plots are not feature evidence. Every required framework must execute successfully.

### 6. Evidence and strict verdict

Collect the final 6. Evidence and strict verdict verdict without discarding earlier failed or blocked required cases.

**Input:** `sdk_metrics`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('6. Evidence and strict verdict', inputs={'sdk_metrics': globals().get('sdk_metrics')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    utils.format_output('6. Evidence and strict verdict', {'value_1': "Actual SDK usage (not estimated):", 'sdk_metrics': sdk_metrics}, status='info', part='RESULT')
    run.finish()


## 7. Opt-in cleanup
Preview and remove only a Foundry agent version newly created by this run. Reused versions, predecessor contracts and connections remain untouched.

### 7. Opt-in cleanup

Apply the explicitly enabled 7. Opt-in cleanup action only to resources owned by this run.

**Input:** `owned_agent_version`, `cleanup_enabled`, `project`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('7. Opt-in cleanup', inputs={'owned_agent_version': globals().get('owned_agent_version'), 'cleanup_enabled': globals().get('cleanup_enabled'), 'project': globals().get('project')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    utils.format_output('7. Opt-in cleanup', {'value_1': "Run-owned Foundry version:", 'owned_agent_version': owned_agent_version}, status='info', part='RESULT')
    if cleanup_enabled and owned_agent_version:
        from azure.identity import DefaultAzureCredential
        from azure.ai.projects import AIProjectClient
        with AIProjectClient(endpoint=owned_agent_version["project_endpoint"], credential=DefaultAzureCredential()) as project:
            project.agents.delete_version(
                agent_name=owned_agent_version["name"], agent_version=owned_agent_version["version"],
            )
    else:
        utils.format_output('7. Opt-in cleanup', {'value_1': "Cleanup disabled or agent version reused; predecessor APIM resources and reused agent versions are retained."}, status='info', part='RESULT')
